# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [18]:
print('Ready. No external packages required for this practical.')

Ready. No external packages required for this practical.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [19]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'i am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [20]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['she', 'is', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
{'?X': ['worried']}


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [21]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [22]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams      -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
The weather is fine today        -> Tell me more about that.


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [23]:
# TODO: your extended RULES table and a test of each new rule
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours', 'feel': 'feel'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

RULES = [
    # Original rules
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
    
    # 5 New Rules:
    (['i', 'feel', '?X'],            'Why do you feel {X}?'),                   
    (['i', 'think', '?X'],           'Do you really think {X}?'),               
    (['i', 'like', '?X'],            'What do you like most about {X}?'),       
    (['hello', '?X'],                'Hello! How can I help you today?'),
    (['who', 'are', 'you', '?X'],    'I am a Muhammad Hashair .')
]

FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            if '?X' in b:
                captured = swap_person(b['?X'])
                return template.format(X=' '.join(captured))
            return template
    return FALLBACK

# Test cases for the 5 new rules
test_sentences = [
    "I feel happy today",
    "I think python is easy",
    "I like artificial intelligence",
    "Hello there",
    "Who are you anyway"
]

print('----HASHAIR TASK 1----')
for s in test_sentences:
    print(f"Input : {s}")
    print(f"Reply : {respond(s)}\n")

----HASHAIR TASK 1----
Input : I feel happy today
Reply : Why do you feel happy today?

Input : I think python is easy
Reply : Do you really think python is easy?

Input : I like artificial intelligence
Reply : What do you like most about artificial intelligence?

Input : Hello there
Reply : Hello! How can I help you today?

Input : Who are you anyway
Reply : I am a Muhammad Hashair .



### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [24]:
# TODO: an improved matcher, plus tests showing it works

def match_middle(pattern, tokens):
    """
    Matches pattern containing single variable ?X at any position.
    Returns dict of bindings, or None if no match.
    """
    # Find position of variable ?X in pattern
    var_index = -1
    for i, token in enumerate(pattern):
        if token.startswith('?'):
            var_index = i
            break

    # If no variable present, match lengths and exact words
    if var_index == -1:
        return {} if pattern == tokens else None

    # Prefix (words before variable) and Suffix (words after variable)
    prefix = pattern[:var_index]
    suffix = pattern[var_index + 1:]

    # Ensure total tokens length is sufficient to cover prefix and suffix
    if len(tokens) < len(prefix) + len(suffix):
        return None

    # Check prefix match
    if tokens[:len(prefix)] != prefix:
        return None

    # Check suffix match
    if suffix and tokens[-len(suffix):] != suffix:
        return None

    # Extract middle tokens captured by ?X
    end_bound = len(tokens) - len(suffix) if suffix else len(tokens)
    captured = tokens[len(prefix):end_bound]

    return {pattern[var_index]: captured}

print("--- HASHAIR Task 2 ---")
p1 = ['i', '?X', 'about', 'exams']
t1 = 'i am worried about exams'.split()
t2 = 'i feel stressed about exams'.split()
t3 = 'i am worried about assignment'.split()

print("Pattern:", p1)
print("Input 1:", t1, "-> Match:", match_middle(p1, t1))
print("Input 2:", t2, "-> Match:", match_middle(p1, t2))
print("Input 3:", t3, "-> Match:", match_middle(p1, t3))

--- HASHAIR Task 2 ---
Pattern: ['i', '?X', 'about', 'exams']
Input 1: ['i', 'am', 'worried', 'about', 'exams'] -> Match: {'?X': ['am', 'worried']}
Input 2: ['i', 'feel', 'stressed', 'about', 'exams'] -> Match: {'?X': ['feel', 'stressed']}
Input 3: ['i', 'am', 'worried', 'about', 'assignment'] -> Match: None


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [25]:
# TODO: represent expressions as nested lists and apply rewrite rules

def simplify_step(expr):
    """Applies basic algebraic rewrite rules to a nested expression."""
    # Base case: constants or variables (not lists) remain unchanged
    if not isinstance(expr, list):
        return expr

    op = expr[0]
    left = simplify_step(expr[1])
    right = simplify_step(expr[2])

    # Rule 1: 1 * x -> x
    if op == '*' and left == 1:
        return right
    # Rule 2: x * 1 -> x
    if op == '*' and right == 1:
        return left
    # Rule 3: 0 + x -> x
    if op == '+' and left == 0:
        return right
    # Rule 4: x + 0 -> x
    if op == '+' and right == 0:
        return left

    return [op, left, right]

def simplify_completely(expr):
    """Applies rules repeatedly until nothing changes."""
    while True:
        new_expr = simplify_step(expr)
        if new_expr == expr:
            return new_expr
        expr = new_expr

expression = ['+', ['*', 1, 'x'], 0]

print("--- Hashair Task 3 ---")
print("Original Expression :", expression)
print("Simplified Result   :", simplify_completely(expression))

--- Hashair Task 3 ---
Original Expression : ['+', ['*', 1, 'x'], 0]
Simplified Result   : x


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [26]:
"""""
Contextual understanding of the ongoing conversation topic.
2. Empathy, real logic, or awareness of psychological concepts.
3. Personal knowledge or facts about the user's life, exams, or feelings.
"""""

'""\nContextual understanding of the ongoing conversation topic.\n2. Empathy, real logic, or awareness of psychological concepts.\n3. Personal knowledge or facts about the user\'s life, exams, or feelings.\n'

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

In [27]:
# What the program actually stores?
"""""
1. List of string : Pre-defined lists of string tokens representing expected input patterns.
2. Fixed string response : coded text containing placeholders (`{X}`).
3. Word swap dictionary: A basic map to swap words
"""

'""\n1. List of string : Pre-defined lists of string tokens representing expected input patterns.\n2. Fixed string response : coded text containing placeholders (`{X}`).\n3. Word swap dictionary: A basic map to swap words\n'

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence               | Rule that matched    | Reply produced                               | Sensible? |
|I feel stressed about my exams|['i', 'feel', '?X']   |Why do you feel stressed about your exams?    |Yes        |
|I think python is easy        | ['i', 'think', '?X'] | Do you really think python is easy?          |Yes        |
| I cannot sleep properly      |['i', 'cannot', '?X'] |What stops you being able to sleep properly?  |yes        |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|-string--                           |-matched word returned--|

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. No, it understands nothing. The program only stores string lists, rule patterns, and substitution dictionary keys. It performs syntax-level string parsing and reassembly without any conceptual model or semantic knowledge of the world.

2. Swapping pronoun person words transforms statement inputs into interactive second-person conversational phrasing. It mirrors real human dialogue dynamics, tricking human brains into inferring comprehension.

3. It shows that human evaluation of conversational fluency is heavily subjective. People easily project intelligence and empathy onto systems purely based on grammatical syntax rather than true underlying knowledge processing.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [28]:
NAME = "MUHAMMAD HASHAIR"
ROLL_NUMBER = "2025-cse-048"
print(NAME, ROLL_NUMBER)

MUHAMMAD HASHAIR 2025-cse-048
